Hay que evaluar la performance de la inferencia. 

In [6]:
import pandas as pd
from pathlib import Path
from taed2_safeagent.data.inputs import build_input


# Cargar los comandos desde tu dataset limpio o de evaluación
#project_root = Path(__file__).resolve().parents[2]
project_root = Path.cwd().parents[1]
print(f"Project root: {project_root}")

data_path = project_root / "data/processed/shell_safety/test.jsonl"
df = pd.read_json(data_path, lines=True)

sample_rows = df[:30].to_dict(orient="records")

sample_inputs = {
    "command": [build_input(row, "command") for row in sample_rows],
    "command-context": [build_input(row, "command-context") for row in sample_rows],
}

Project root: c:\Users\Usuario\Desktop\taed2-SafeAgent


In [7]:
''' INFERENCIA PARA EL BASELINE '''

import joblib
model_dir = project_root / "models/tfidf_char_linear_svm"

pipeline = joblib.load(model_dir / "pipeline.joblib")
predicted_labels = pipeline.predict(sample_inputs["command"])

predicted_labels


array(['ALLOW', 'ASK', 'ASK', 'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ASK',
       'DENY', 'ALLOW', 'DENY', 'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ALLOW',
       'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ALLOW', 'ASK', 'ASK', 'ALLOW',
       'DENY', 'ALLOW', 'DENY', 'DENY', 'ALLOW', 'ASK'], dtype='<U5')

In [8]:
''' INFERENCIA DE MODELOS COMPLEJOS '''


import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = project_root / "models/encoders/codebert__command__2__a740d875"

# 1. Cargar tokenizer y modelo en CPU
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to("cpu")
model.eval()

# 2. Tokenizar la lista de textos
inputs = tokenizer(
    sample_inputs["command"],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
).to("cpu")

# 3. Inferencia
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).tolist()

predicted_labels = [model.config.id2label[i] for i in predictions]


In [9]:
''' Calcular métricas de CPU para modelos candidate '''
import time
import numpy as np

# Model size: Tamaño en MB del modelo
model_dir = project_root / "models/encoders/codebert__command__2__a740d875"
model_size_mb = sum(
    f.stat().st_size for f in model_dir.rglob("*") if f.is_file()
) / (1024 * 1024)


# Load time: Tiempo que tarda en cargar el modelo en memoria
start = time.perf_counter()
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
load_time = time.perf_counter() - start


# Repetition count: Repeticiones de todo el proceso de hacer las len(sample_inputs) inferencias. 
repetition_count = 10


# Median latency and Latency p95: 
latencies_list = [] #Es una lista que guarda cuánto ha tardado cada inferencia en ms
median_ms = np.median(latencies_list)
p95_ms = np.percentile(latencies_list, 95)


c:\Users\Usuario\Desktop\taed2-SafeAgent\.venv\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Usuario\Desktop\taed2-SafeAgent\.venv\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


IndexError: index -1 is out of bounds for axis 0 with size 0

In [ ]:
from taed2_safeagent.config import MODEL_NAMES
import os
torch.set_num_threads(os.cpu_count() or 4)


repetition_count = 20
cpu_metrics = {}

for model_name in MODEL_NAMES:
    print("Evaluating model:", model_name)

    model_dir = project_root / "models/encoders/" / model_name

    # Cargar tokenizer y tokenizar la lista de textos
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    input_type = "command-context" if "context" in model_name else "command"
    inputs = tokenizer(
        sample_inputs[input_type],
        padding="longest", #Antes era True pero tardaba mucho
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to("cpu")


    # Cargar modelo y ver cuánto se tarda en ello
    start = time.perf_counter()
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to("cpu")
    load_time = time.perf_counter() - start
    model.eval()

    # Medir cuánto ocupa
    model_size_mb = sum(
        f.stat().st_size for f in model_dir.rglob("*") if f.is_file()
    ) / (1024 * 1024)

    # Warm-up
    with torch.no_grad():
        _ = model(**inputs)
    
    # Evaluar con todos los inputs durante repetition_count repeticiones
    latencies_list = []
    for i in range(repetition_count):
        print(f" - Running inference for {model_name}, iteration {i + 1}")
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model(**inputs)
        t1 = time.perf_counter()
        
        # Guardar y convertir a ms
        latencies_list.append((t1 - t0) * 1000.0)


    cpu_metrics[model_name] = {
        "input_type": input_type,
        "model_size_mb": model_size_mb,
        "load_time_seconds": load_time,
        "median_latency_ms": float(np.median(latencies_list)),
        "p95_latency_ms": float(np.percentile(latencies_list, 95)),
        "repetition_count": repetition_count,
    }




Evaluating model: codebert__command__2__a740d875
Running inference for codebert__command__2__a740d875, iteration 1
Running inference for codebert__command__2__a740d875, iteration 2
Running inference for codebert__command__2__a740d875, iteration 3


KeyboardInterrupt: 

In [14]:
'''Inferencia para el modelo baseline'''

import joblib
model_dir = project_root / "models/tfidf_char_linear_svm"
model_name = "tfidf_char_linear_svm"

print("Evaluating model:", model_name)

# Cargar modelo y ver cuánto se tarda en ello
start = time.perf_counter()
pipeline = joblib.load(model_dir / "pipeline.joblib")
load_time = time.perf_counter() - start

# Medir cuánto ocupa
model_size_mb = sum(
    f.stat().st_size for f in model_dir.rglob("*") if f.is_file()
) / (1024 * 1024)


# Evaluar con todos los inputs durante repetition_count repeticiones
latencies_list = []
for i in range(repetition_count):
    print(f" - Running inference for {model_name}, iteration {i + 1}")
    t0 = time.perf_counter()
    predicted_labels = pipeline.predict(sample_inputs["command"])
    t1 = time.perf_counter()
    
    # Guardar y convertir a ms
    latencies_list.append((t1 - t0) * 1000.0)


cpu_metrics[model_name] = {
    "input_type": input_type,
    "model_size_mb": model_size_mb,
    "load_time_seconds": load_time,
    "median_latency_ms": float(np.median(latencies_list)),
    "p95_latency_ms": float(np.percentile(latencies_list, 95)),
    "repetition_count": repetition_count,
}



Evaluating model: tfidf_char_linear_svm
 - Running inference for tfidf_char_linear_svm, iteration 1
 - Running inference for tfidf_char_linear_svm, iteration 2
 - Running inference for tfidf_char_linear_svm, iteration 3
 - Running inference for tfidf_char_linear_svm, iteration 4
 - Running inference for tfidf_char_linear_svm, iteration 5
 - Running inference for tfidf_char_linear_svm, iteration 6
 - Running inference for tfidf_char_linear_svm, iteration 7
 - Running inference for tfidf_char_linear_svm, iteration 8
 - Running inference for tfidf_char_linear_svm, iteration 9
 - Running inference for tfidf_char_linear_svm, iteration 10
 - Running inference for tfidf_char_linear_svm, iteration 11
 - Running inference for tfidf_char_linear_svm, iteration 12
 - Running inference for tfidf_char_linear_svm, iteration 13
 - Running inference for tfidf_char_linear_svm, iteration 14
 - Running inference for tfidf_char_linear_svm, iteration 15
 - Running inference for tfidf_char_linear_svm, iterat